# Ungraded Activity: Let's build a simple replay buffer
In this activity, we'll build a replay buffer that will allow a hypothetical planet exploration robot to store and playback its movement on a planet's surface using a coupled queue and stack.

> __Scenario:__ A surveyor robot is exploring the surface of a distant planet. As it moves, it records its starting position and the actions (moves) it takes at each time step. The robot can move north, south, east, or west. It can return to a previous position by reversing its last action. The robot needs a way to store its movements so that it can later replay them, either to retrace its steps to its homebase, e.g., charging station, or to correct mistakes.

Need to fill in a one sentence descrription of the activity here. 

Let's get stated!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.
* The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

In [54]:
include("Include.jl");

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material. 

### Constants
Let's define some constants that we will use throughout the activity. See that comment next to the constant for it's meaning, permissible values, units, etc.

In [55]:
number_of_moves_per_episode = 1000; # How many moves the robot will make in each episode
actions = Dict(1 => 'n', 2 => 's', 3 => 'e', 4 => 'w', 5 => 'p', 6 => 'b'); # The actions the robot can take, where 1 = north, 2 = south, 3 = east, and 4 = west
moves = Dict('n' => (0, 1), 's' => (0, -1), 'e' => (1, 0), 'w' => (-1, 0), 'p' => (0, 0), 'b' => (0, 0)); # The moves the robot can make
starting_position = (0, 0); # The starting position of the robot (the grid position of the charging station)

___

## Task 1: Transmit the robot's movements to a queue
In this task, we we generate a series of random movements for the robot and transmit them to a queue. The robot will make a series of moves, and we will store the starting position and the action taken at each time step in the queue.

From earth, we send a vector of integers representing the actions to be taken by the robot, the robot stores the corresponding characters in the queue. Let's generate a random sequence of commands for the robot to execute in the `commands_to_execute::Array{Int,1}` vector.

In [56]:
commands_to_execute = let

    # initialize - 
    commands = Array{Int,1}(undef, number_of_moves_per_episode);
    p = [0.2, 0.1, 0.3, 0.1, 0.3]; # The probabilities of each action being taken
    d = Categorical(p); # The categorical distribution of the actions
    θ = 0.20; # Fraction of the time we transmit a "back" command to the robot

    # generate a random command sequence -
    for i in 1:number_of_moves_per_episode
        if rand() < θ && i > 1 # if the random number is less than θ and not the first command
            commands[i] = 6; # back command
        else
            commands[i] = rand(d); # one of {n,s,e,w,pause}
        end
    end

    commands; # return the commands
end

1000-element Vector{Int64}:
 3
 3
 6
 6
 6
 3
 5
 1
 5
 3
 ⋮
 1
 2
 2
 6
 3
 6
 2
 6
 6

__Check:__ Let's check that the `commands_to_execute::Array{Int,1}` vector is valid, i.e., it has the correct length, contains only valid commands and does not start with a `back` command.

There are a couple of ways to do this test, one is to use [the `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) (which we've been doing) to check that the output of our bubble sort implementation matches the output of [the built-in Julia `sort(...)` method](https://docs.julialang.org/en/v1/base/sort/#Base.sort).

> __Alternative:__ When we have multiple tests to run, we can use [the `Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/) to write __unit tests__ for our command sequence implementation. The [`Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/) provides a framework for writing and running tests in Julia, including support for assertions, test cases, and test suites.

In this case, we can use the `@testset` macro to group our tests together and the `@test` macro to check that the output of our command sequence implementation matches the expected output.

In [57]:
let
    @testset "Valid Command Sequence" begin
        @test commands_to_execute isa Array{Int,1}; # ensure commands_to_execute is an array of integers
        @test all(c -> c ∈ keys(actions), commands_to_execute); # ensure all commands are valid
        @test commands_to_execute[1] != 6; # ensure the first command is not a back command (will this ever be true?)
    end
end;

Test Summary:          | Pass  Total  Time
Valid Command Sequence |    3      3  0.0s


If all the tests pass, we can be confident that the `commands_to_execute::Array{Int,1}` vector is valid and ready to be transmitted to the robot. If we have a failed test, we need to regenerate the `commands_to_execute::Array{Int,1}` vector until all tests pass.

__All tests have passed.__ So the commands stored in the `commands_to_execute::Array{Int,1}` have been transmitted to the robot! However, becuase the robot was build by different government subcontractors, the robot's software is not able to interpret the commands directly. Instead, we need to convert the commands to characters that the robot can understand, we'll then store the characters in the `command_queue::Queue{Char}`.

In [58]:
command_queue = let

    # initialize -
    q = Queue{Char}();
    for c ∈ commands_to_execute
        enqueue!(q, actions[c]); # convert the command to a character and push it to the queue
    end
    q; # return the queue
end

Queue{Char}(Deque [['e', 'e', 'b', 'b', 'b', 'e', 'p', 'n', 'p', 'e', 'p', 'b', 'p', 'n', 'b', 'w', 'p', 'n', 'e', 'e', 'e', 'b', 'b', 'n', 'n', 'n', 'b', 'e', 'b', 'w', 'w', 'e', 'w', 'p', 'p', 'b', 'p', 'b', 'e', 'e', 's', 'n', 'e', 'n', 'p', 'e', 'p', 'p', 'w', 'n', 'b', 'p', 'e', 'n', 'p', 'b', 'n', 'n', 'p', 'p', 'p', 's', 'p', 'n', 'e', 'b', 's', 'b', 'n', 'p', 'w', 'w', 'e', 'e', 'n', 'b', 'w', 'p', 'p', 'b', 'e', 'w', 'w', 'p', 'b', 'w', 'p', 'b', 'n', 'e', 'e', 'n', 'b', 'b', 's', 'e', 'b', 'p', 'e', 'p', 'e', 'e', 'p', 'e', 'p', 'e', 'e', 'n', 'w', 'b', 'p', 'n', 'p', 'b', 'b', 'p', 'p', 'p', 'p', 'e', 'p', 'e', 'e', 'p', 'b', 'p', 'n', 'e', 'b', 'w', 'p', 'e', 'e', 'e', 'b', 'w', 'p', 'b', 'n', 'b', 'e', 's', 'e', 'b', 'e', 'p', 'e', 'n', 'p', 'b', 's', 'w', 'e', 'b', 'p', 'n', 'w', 'b', 'p', 'p', 'b', 'n', 'e', 'p', 'n', 'b', 'p', 'e', 's', 'p', 'b', 'p', 'p', 'b', 'e', 'p', 'e', 'b', 'b', 'p', 'p', 'p', 'w', 'n', 'e', 'e', 'n', 'p', 'p', 'n', 'b', 'e', 'b', 'e', 'e', 'b', 

## Task 2: Execute the commands in the queue
In this task, we will execute the commands in the `command_queue::Queue{Char}`. The robot will start at the `starting_position::Tuple{Int,Int}` and will execute the commands in the queue one by one. The robot will move in the direction specified by the command, and will update its position accordingly. 

> __What happens if the robot gets a `back` command?__ The robot will reverse its last action, i.e., it will move back to the previous position. If the robot is at the starting position, it will not move. We will use a `Stack{Char}` to keep track of the robot's last executed command, so that we can reverse it if needed. Does this seem familiar? Yes, it's the essense of undo/redo functionality in many applications, where the last action can be reversed or redone.

However, before we do that, we need to think about the inverse of an action. For example, if the robot moves north, and the next action is a `back` command, the robot will move back to its previous position by going south. Likewise, if the robot moves west, it can move east to return to its previous position. Let's build a dictionary that maps the actions to their inverse actions, so that we can easily reverse the robot's last action if needed.

In [59]:
inverse_action = Dict('n' => 's', 's' => 'n', 'e' => 'w', 'w' => 'e', 'p' => 'p', 'b' => 'b'); # The inverse actions of the robot

Ok, so now we are ready to wander around the planet! We'll track the robot's position in the `robot_position_dict::Dict{Int64, Tuple{Int,Int}}` dictionary, where the key is the time step and the value is the position of the robot at that time step. 

In [ ]:
robot_position_dict = let

    # initialize -
    robot_position = Dict{Int64, Tuple{Int,Int}}();
    robot_position[0] = starting_position; # the robot starts at the charging station
    undo = Stack{Char}(); # stack to keep track of the robot's last action
    counter = 1; # counter for the time step
    q = command_queue; # the command queue

    # execute the commands in the queue -
    while isempty(q) == false

        # get the current robot position
        xᵢ = robot_position[counter-1][1];
        yᵢ = robot_position[counter-1][2];
        
        # get the next command from the queue
        command = dequeue!(q); # get the next command from the queue

        # process the command       
        action = nothing;
        if command == 'b' # if the command is a back command
            if isempty(undo) == false # if there is an action to undo
                action = pop!(undo) |> a-> inverse_action[a] # get the last action from the stack
            else
                @assert false "No action to undo"; # if there is no action to undo, raise an error
            end
        else # if the command is not a back command
            push!(undo, command); # push the command to the stack
        end

        action = action === nothing ? command : action; # if the action is not set, use the command as the action
        Δ = moves[action]; # get the move associated with the action
        xᵢ += Δ[1]; # update the x position
        yᵢ += Δ[2]; # update the y position
        
        
        robot_position[counter] = (xᵢ, yᵢ); # update the robot position in the dictionary
        counter += 1; # increment the time step counter
    end

    robot_position;
end

MethodError: MethodError: no method matching copy(::Queue{Char})
The function `copy` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  copy(!Matched::DataFrames.Index)
   @ DataFrames ~/.julia/packages/DataFrames/kcA9R/src/other/index.jl:39
  copy(!Matched::Pkg.Resolve.GraphData)
   @ Pkg ~/.julia/juliaup/julia-1.11.6+0.aarch64.apple.darwin14/share/julia/stdlib/v1.11/Pkg/src/Resolve/graphtype.jl:162
  copy(!Matched::BenchmarkGroup)
   @ BenchmarkTools ~/.julia/packages/BenchmarkTools/1i1mY/src/groups.jl:55
  ...


In [61]:
robot_position_dict

Dict{Int64, Tuple{Int64, Int64}} with 1 entry:
  0 => (0, 0)